# setup

In [1]:
%%capture
import os, re
if "COLAB_" not in "".join(os.environ.keys()):
    !uv pip install unsloth  # Do this in local & cloud setups
else:
    import torch; v = re.match(r'[\d]{1,}\.[\d]{1,}', str(torch.__version__)).group(0)
    xformers = 'xformers==' + {'2.10':'0.0.34','2.9':'0.0.33.post1','2.8':'0.0.32.post2'}.get(v, "0.0.34")
    !uv pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
    !uv pip install --no-deps unsloth_zoo bitsandbytes accelerate {xformers} peft trl triton unsloth
    !uv pip install --no-deps --upgrade "torchao>=0.16.0"
!uv pip install --no-deps transformers==5.5.0 "tokenizers>=0.22.0,<=0.23.0"
!uv pip install torchcodec
import torch; torch._dynamo.config.recompile_limit = 64;

In [2]:
!uv pip install datasets

Using Python 3.12.13 environment at: /usr
Checked 1 package in 129ms


In [3]:
from unsloth import FastModel
import torch


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


In [4]:
gemma4_models = [
    # Gemma-4 instruct models:
    "unsloth/gemma-4-E2B-it",
    "unsloth/gemma-4-E4B-it",
    "unsloth/gemma-4-31B-it",
    "unsloth/gemma-4-26B-A4B-it",
    # Gemma-4 base models:
    "unsloth/gemma-4-E2B",
    "unsloth/gemma-4-E4B",
    "unsloth/gemma-4-31B",
    "unsloth/gemma-4-26B-A4B",
] # More models at https://huggingface.co/unsloth


Select the base model

In [5]:

model, tokenizer = FastModel.from_pretrained(
    model_name = "unsloth/gemma-4-E2B",
    dtype = None, # None for auto detection
    max_seq_length = 8192, # Choose any for long context!
    load_in_4bit = True,  # 4 bit quantization to reduce memory
    full_finetuning = False, # [NEW!] We have full finetuning now!
    #device_map = "balanced", # Uses 2x Tesla T4s

    # token = "YOUR_HF_TOKEN", # HF Token for gated models
)

==((====))==  Unsloth 2026.6.9: Fast Gemma4 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.562 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.34. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/2011 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/181 [00:00<?, ?B/s]

processor_config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/32.2M [00:00<?, ?B/s]

# get chat template

In [6]:
from unsloth.chat_templates import get_chat_template
tokenizer = get_chat_template(
    tokenizer,
    chat_template = "gemma-4-thinking",
     #chat_template = "gemma-4"
)

In [7]:
from transformers import TextStreamer
# Helper function for inference
def do_gemma_4_inference(messages, max_new_tokens = 128):
    _ = model.generate(
        **tokenizer.apply_chat_template(
            messages,
            add_generation_prompt = True, # Must add for generation
            tokenize = True,
            return_dict = True,
            return_tensors = "pt",
        ).to("cuda"),
        max_new_tokens = max_new_tokens,
        use_cache = True,
        temperature = 1.0, top_p = 0.95, top_k = 64,
        streamer = TextStreamer(tokenizer, skip_prompt = True),
    )

In [8]:
print(tokenizer.chat_template)

{{ bos_token }}{%- macro strip_thinking(text) -%}
    {%- set ns = namespace(result='') -%}
    {%- for part in text.split('<channel|>') -%}
        {%- if '<|channel>' in part -%}
            {%- set ns.result = ns.result + part.split('<|channel>')[0] -%}
        {%- else -%}
            {%- set ns.result = ns.result + part -%}
        {%- endif -%}
    {%- endfor -%}
    {{- ns.result | trim -}}
{%- endmacro -%}
{%- set thinking = enable_thinking is defined and enable_thinking -%}
{%- set loop_messages = messages -%}
{%- if messages[0]['role'] in ['system', 'developer'] or thinking -%}
    {{ '<|turn>system
' }}
    {%- if thinking -%}
        {{ '<|think|>
' }}
    {%- endif -%}
    {%- if messages[0]['role'] in ['system', 'developer'] -%}
        {{ messages[0]['content'] | trim }}
        {%- set loop_messages = messages[1:] -%}
    {%- endif -%}
    {{ '<turn|>
' }}
{%- endif -%}
{%- for message in loop_messages -%}
    {%- if (message['role'] == 'user') != (loop.index0 % 2 == 0)

In [9]:
messages = [{
    "role": "user",
    "content": [{ "type" : "text",
                  "text" : "The capital of Italy is" }]
}]
do_gemma_4_inference(messages, max_new_tokens = 32)

ing
The capital of Italy is Rome.
The capital of Italy is Rome.
The capital of Italy is Rome.
The capital of Italy is Rome


In [10]:
print("messages:", messages)
print("\nchat template:\n", tokenizer.apply_chat_template(
            messages,
            add_generation_prompt = True, # Must add for generation
        ))

messages: [{'role': 'user', 'content': [{'type': 'text', 'text': 'The capital of Italy is'}]}]

chat template:
 <bos><|turn>user
The capital of Italy is<turn|>
<|turn>model
<|channel>thought
<channel|>


# Create SFT dataset

In [11]:
# mix of dataset

In [12]:
from datasets import load_dataset

In [13]:
alpaca_dataset = load_dataset('vicgalle/alpaca-gpt4', split='train[:10000]')

README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00001-6ef3991c06080e(…):   0%|          | 0.00/48.4M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/52002 [00:00<?, ? examples/s]

In [14]:
alpaca_dataset

Dataset({
    features: ['instruction', 'input', 'output', 'text'],
    num_rows: 10000
})

In [15]:
finetome_dataset = load_dataset("mlabonne/FineTome-100k", split = "train[:3000]")

README.md:   0%|          | 0.00/982 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/117M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/100000 [00:00<?, ? examples/s]

In [16]:
finetome_dataset

Dataset({
    features: ['conversations', 'source', 'score'],
    num_rows: 3000
})

In [17]:
everyday_convos = load_dataset("HuggingFaceTB/everyday-conversations-llama3.1-2k", split='train_sft')

README.md: 0.00B [00:00, ?B/s]

data/train_sft-00000-of-00001.parquet:   0%|          | 0.00/2.05M [00:00<?, ?B/s]

data/test_sft-00000-of-00001.parquet:   0%|          | 0.00/124k [00:00<?, ?B/s]

Generating train_sft split:   0%|          | 0/2260 [00:00<?, ? examples/s]

Generating test_sft split:   0%|          | 0/119 [00:00<?, ? examples/s]

In [18]:
everyday_convos

Dataset({
    features: ['topic', 'subtopic', 'subsubtopic', 'full_topic', 'prompt', 'completion', 'token_length', 'messages'],
    num_rows: 2260
})

In [19]:
everyday_convos[0]['messages']

[{'content': 'Hey!', 'role': 'user'},
 {'content': 'Hello! How can I help you today?', 'role': 'assistant'},
 {'content': "I'm trying to track my expenses. Can you help me with that?",
  'role': 'user'},
 {'content': 'Yes, I can help you track your expenses. You can start by telling me your income and fixed expenses, such as rent and utilities.',
  'role': 'assistant'},
 {'content': "That sounds easy. How do I know if I'm staying within my budget?",
  'role': 'user'},
 {'content': "Once you've entered your income and expenses, I can help you set a budget and alert you when you're going over. You can also categorize your spending to see where your money is going.",
  'role': 'assistant'},
 {'content': 'Okay, that sounds great. Can you remind me to review my budget regularly?',
  'role': 'user'},
 {'content': 'I can send you reminders to review your budget weekly, monthly, or at any interval you prefer.',
  'role': 'assistant'}]

# formatting datasets for SFT

In [20]:
alpaca_dataset[0]

{'instruction': 'Give three tips for staying healthy.',
 'input': '',
 'output': '1. Eat a balanced and nutritious diet: Make sure your meals are inclusive of a variety of fruits and vegetables, lean protein, whole grains, and healthy fats. This helps to provide your body with the essential nutrients to function at its best and can help prevent chronic diseases.\n\n2. Engage in regular physical activity: Exercise is crucial for maintaining strong bones, muscles, and cardiovascular health. Aim for at least 150 minutes of moderate aerobic exercise or 75 minutes of vigorous exercise each week.\n\n3. Get enough sleep: Getting enough quality sleep is crucial for physical and mental well-being. It helps to regulate mood, improve cognitive function, and supports healthy growth and immune function. Aim for 7-9 hours of sleep each night.',
 'text': 'Below is an instruction that describes a task. Write a response that appropriately completes the request.\n\n### Instruction:\nGive three tips for 

In [21]:
alpaca_dataset[0]['instruction']

'Give three tips for staying healthy.'

In [22]:
print(alpaca_dataset[0]['output'])

1. Eat a balanced and nutritious diet: Make sure your meals are inclusive of a variety of fruits and vegetables, lean protein, whole grains, and healthy fats. This helps to provide your body with the essential nutrients to function at its best and can help prevent chronic diseases.

2. Engage in regular physical activity: Exercise is crucial for maintaining strong bones, muscles, and cardiovascular health. Aim for at least 150 minutes of moderate aerobic exercise or 75 minutes of vigorous exercise each week.

3. Get enough sleep: Getting enough quality sleep is crucial for physical and mental well-being. It helps to regulate mood, improve cognitive function, and supports healthy growth and immune function. Aim for 7-9 hours of sleep each night.


In [23]:
# format alpaca to match the chat template format
def create_conversation_alpaca(example):
    return {
        "messages": [
            {"role": "user", "content": example['instruction']},
            {"role": "assistant", "content": example['output']},
        ]
    }

In [24]:
alpaca_dataset

Dataset({
    features: ['instruction', 'input', 'output', 'text'],
    num_rows: 10000
})

In [25]:
alpaca_dataset[100]

{'instruction': 'Design a database to record employee salaries.',
 'input': '',
 'output': "Here is a suggested design for a database to record employee salaries:\n\n1. **Employee Table**: This table will store all the relevant information about an employee. Some of the fields in this table could include:\n\n- Employee ID: An unique identifier for each employee.\n- First Name: The employee's first name.\n- Last Name: The employee's last name.\n- Email: The employee's email address.\n- Hire Date: The date the employee was hired.\n- Department: The department the employee works in.\n\n2. **Salary Table**: This table will store all the relevant information about an employee's salary. Some of the fields in this table could include:\n\n- Salary ID: An unique identifier for each salary record\n- Employee ID: The employee this salary record is for; this field should be a foreign key that references the Employee table.\n- Salary Amount: The amount of the employee's salary.\n- Start Date: The d

In [26]:
alpaca_dataset = alpaca_dataset.map(create_conversation_alpaca, batched=False)

Map:   0%|          | 0/10000 [00:00<?, ? examples/s]

In [27]:
alpaca_dataset['messages'][0]

[{'role': 'user', 'content': 'Give three tips for staying healthy.'},
 {'role': 'assistant',
  'content': '1. Eat a balanced and nutritious diet: Make sure your meals are inclusive of a variety of fruits and vegetables, lean protein, whole grains, and healthy fats. This helps to provide your body with the essential nutrients to function at its best and can help prevent chronic diseases.\n\n2. Engage in regular physical activity: Exercise is crucial for maintaining strong bones, muscles, and cardiovascular health. Aim for at least 150 minutes of moderate aerobic exercise or 75 minutes of vigorous exercise each week.\n\n3. Get enough sleep: Getting enough quality sleep is crucial for physical and mental well-being. It helps to regulate mood, improve cognitive function, and supports healthy growth and immune function. Aim for 7-9 hours of sleep each night.'}]

format finetome

In [28]:
finetome_dataset[0]

{'conversations': [{'from': 'human',
   'value': 'Explain what boolean operators are, what they do, and provide examples of how they can be used in programming. Additionally, describe the concept of operator precedence and provide examples of how it affects the evaluation of boolean expressions. Discuss the difference between short-circuit evaluation and normal evaluation in boolean expressions and demonstrate their usage in code. \n\nFurthermore, add the requirement that the code must be written in a language that does not support short-circuit evaluation natively, forcing the test taker to implement their own logic for short-circuit evaluation.\n\nFinally, delve into the concept of truthiness and falsiness in programming languages, explaining how it affects the evaluation of boolean expressions. Add the constraint that the test taker must write code that handles cases where truthiness and falsiness are implemented differently across different programming languages.'},
  {'from': 'gpt

In [29]:
from unsloth.chat_templates import standardize_data_formats
finetome_dataset = standardize_data_formats(finetome_dataset)

Unsloth: Standardizing formats (num_proc=8):   0%|          | 0/3000 [00:00<?, ? examples/s]

In [30]:
finetome_dataset[15]

{'conversations': [{'role': 'system',
   'content': 'You are a helpful assistant, with no access to external functions.'},
  {'role': 'user',
   'content': 'Explain the comparison operator in programming.'},
  {'role': 'assistant',
   'content': 'In programming, a comparison operator is used to compare two values and determine if they are equal or if one is greater or less than the other. It is usually denoted by symbols such as "==", ">", "<", ">=", "<=", or "!=". \n\nThe "==" symbol is used to check if two values are equal, while "!=" checks if they are not equal. The ">" symbol checks if the left value is greater than the right value, while "<" checks if the left value is less than the right value. The ">=" symbol checks if the left value is greater than or equal to the right value, while "<=" checks if the left value is less than or equal to the right value.\n\nComparison operators are often used in decision-making structures such as if statements and loops to determine which code 

# Apply chat template

## Alpaca dataset

In [31]:
alpaca_dataset[10]

{'instruction': 'How did Julius Caesar die?',
 'input': '',
 'output': "Julius Caesar, the Roman Military general, and statesman was assassinated on March 15, 44 BC. He was stabbed to death at a meeting of the Senate in the Theatre of Pompey by a group of senators who called themselves the Liberatores. Led by Gaius Cassius Longinus, and Marcus Junius Brutus, they believed that Caesar's dictatorship was subverting the Roman Republic and sought to restore the traditional republican government. The exact number of assailants is not known, but it is believed that a group of as many as 60 conspirators participated in the assassination, and each of them took turns in stabbing Caesar. The day of his death is still known as the Ides of March.",
 'text': "Below is an instruction that describes a task. Write a response that appropriately completes the request.\n\n### Instruction:\nHow did Julius Caesar die?\n\n### Response:\nJulius Caesar, the Roman Military general, and statesman was assassinat

In [32]:
def format_alpaca_dataset(example):
    convos = example['messages']
    texts = [tokenizer.apply_chat_template(convo, tokenize = False, add_generation_prompt = False).removeprefix('<bos>') for convo in convos]
    return { "text" : texts, }

In [33]:
alpaca_txt = alpaca_dataset.map(format_alpaca_dataset, batched=True)

Map:   0%|          | 0/10000 [00:00<?, ? examples/s]

In [34]:
print(alpaca_txt['text'][10])

<|turn>user
How did Julius Caesar die?<turn|>
<|turn>model
Julius Caesar, the Roman Military general, and statesman was assassinated on March 15, 44 BC. He was stabbed to death at a meeting of the Senate in the Theatre of Pompey by a group of senators who called themselves the Liberatores. Led by Gaius Cassius Longinus, and Marcus Junius Brutus, they believed that Caesar's dictatorship was subverting the Roman Republic and sought to restore the traditional republican government. The exact number of assailants is not known, but it is believed that a group of as many as 60 conspirators participated in the assassination, and each of them took turns in stabbing Caesar. The day of his death is still known as the Ides of March.<turn|>



## Finetome txt dataset

In [35]:
finetome_dataset[0]

{'conversations': [{'role': 'user',
   'content': 'Explain what boolean operators are, what they do, and provide examples of how they can be used in programming. Additionally, describe the concept of operator precedence and provide examples of how it affects the evaluation of boolean expressions. Discuss the difference between short-circuit evaluation and normal evaluation in boolean expressions and demonstrate their usage in code. \n\nFurthermore, add the requirement that the code must be written in a language that does not support short-circuit evaluation natively, forcing the test taker to implement their own logic for short-circuit evaluation.\n\nFinally, delve into the concept of truthiness and falsiness in programming languages, explaining how it affects the evaluation of boolean expressions. Add the constraint that the test taker must write code that handles cases where truthiness and falsiness are implemented differently across different programming languages.'},
  {'role': 'as

In [36]:
def format_finetome_dataset(example):
    convos = example['conversations']
    texts = [tokenizer.apply_chat_template(convo, tokenize = False, add_generation_prompt = False).removeprefix('<bos>') for convo in convos]
    return { "text" : texts, }

In [37]:
txt_finetome = finetome_dataset.map(format_finetome_dataset, batched=True)

Map:   0%|          | 0/3000 [00:00<?, ? examples/s]

In [38]:
print(txt_finetome['text'][20])

<|turn>user
Write a program to find the number of letters in each word of a sentence using the map function.<turn|>
<|turn>model
Certainly! Here's a code example that uses the `map()` function to find the number of letters in each word of a sentence:

```python
def count_letters(sentence):
    """
    This function takes a sentence as input and returns a list of the number of letters
    in each word using the map function.

    Args:
        sentence (str): The input sentence containing words.

    Returns:
        list: A list of integers representing the number of letters in each word.

    Examples:
        >>> count_letters("Hello world")
        [5, 5]
        >>> count_letters("Python is awesome")
        [6, 2, 7]
    """
    # Split the sentence into words
    words = sentence.split()

    # Use map to apply len() function on each word and get the length
    letter_counts = list(map(len, words))

    return letter_counts
```

Now, let me explain the code:

1. The `count_letter

## everyday convos

In [39]:
everyday_convos[0]

{'topic': 'Shopping',
 'subtopic': 'Budgeting',
 'subsubtopic': 'Tracking expenses',
 'full_topic': 'Shopping/Budgeting/Tracking expenses',
 'prompt': 'Generate a very simple multi-turn conversation between a User and an AI Assistant about Shopping/Budgeting/Tracking expenses. The conversation should start with a basic greeting like "Hello" or "Hi" and be straightforward. Include 3-4 short exchanges. The AI should give brief, clear answers. The User should ask simple questions.\n\nStart the conversation like this:\n\nUser: [Greeting]\n\nAI: Hello! How can I help you today?\n\nUser: [Continue with a simple question or statement]\n\nAI: [Respond briefly and clearly]\n\nUser: [Ask a follow-up question or make another simple statement]\n\nAI: [Provide a final helpful response]\n\nMake sure the entire conversation remains very simple and easy to understand, focusing on basic topics or requests.',
 'completion': "User: Hi\n\nAI: Hello! How can I help you today?\n\nUser: I'm trying to track m

In [40]:
def format_everyday_convos(example):
    convos = example['messages']
    texts = [tokenizer.apply_chat_template(convo, tokenize = False, add_generation_prompt = False).removeprefix('<bos>') for convo in convos]
    return { "text" : texts, }

In [41]:
txt_everyday = everyday_convos.map(format_everyday_convos, batched=True)

Map:   0%|          | 0/2260 [00:00<?, ? examples/s]

In [42]:
print(txt_everyday['text'][0])

<|turn>user
Hey!<turn|>
<|turn>model
Hello! How can I help you today?<turn|>
<|turn>user
I'm trying to track my expenses. Can you help me with that?<turn|>
<|turn>model
Yes, I can help you track your expenses. You can start by telling me your income and fixed expenses, such as rent and utilities.<turn|>
<|turn>user
That sounds easy. How do I know if I'm staying within my budget?<turn|>
<|turn>model
Once you've entered your income and expenses, I can help you set a budget and alert you when you're going over. You can also categorize your spending to see where your money is going.<turn|>
<|turn>user
Okay, that sounds great. Can you remind me to review my budget regularly?<turn|>
<|turn>model
I can send you reminders to review your budget weekly, monthly, or at any interval you prefer.<turn|>



Concatenate all the txt dataset

In [43]:
txt_list = [alpaca_txt['text'], txt_finetome['text'], txt_everyday['text']]

In [44]:
txt_list 

[Column(['<|turn>user\nGive three tips for staying healthy.<turn|>\n<|turn>model\n1. Eat a balanced and nutritious diet: Make sure your meals are inclusive of a variety of fruits and vegetables, lean protein, whole grains, and healthy fats. This helps to provide your body with the essential nutrients to function at its best and can help prevent chronic diseases.\n\n2. Engage in regular physical activity: Exercise is crucial for maintaining strong bones, muscles, and cardiovascular health. Aim for at least 150 minutes of moderate aerobic exercise or 75 minutes of vigorous exercise each week.\n\n3. Get enough sleep: Getting enough quality sleep is crucial for physical and mental well-being. It helps to regulate mood, improve cognitive function, and supports healthy growth and immune function. Aim for 7-9 hours of sleep each night.<turn|>\n', '<|turn>user\nWhat are the three primary colors?<turn|>\n<|turn>model\nThe three primary colors are red, blue, and yellow. These colors are called p

In [45]:
from datasets import Dataset

In [46]:
from datasets import Dataset, concatenate_datasets

combined_dataset = concatenate_datasets([
    Dataset.from_dict({"text": alpaca_txt['text']}),
    Dataset.from_dict({"text": txt_finetome['text']}),
    Dataset.from_dict({"text": txt_everyday['text']}),
])
combined_dataset

Dataset({
    features: ['text'],
    num_rows: 15260
})

In [47]:
combined_dataset.save_to_disk("../gemma_data/gemma_sft_dataset")

Saving the dataset (0/1 shards):   0%|          | 0/15260 [00:00<?, ? examples/s]

In [48]:
combined_dataset = combined_dataset.shuffle(seed=42)

In [49]:
print(combined_dataset[100]['text'])

<|turn>user
Generate a job description for a front-end developer.<turn|>
<|turn>model
Job Title: Front-End Developer

Job Summary: 
We are seeking a skilled front-end developer to join our team in creating cutting-edge websites and web applications. The ideal candidate will have a strong understanding of user interface design principles, proficiency with JavaScript, HTML, and CSS, and experience working with front-end frameworks such as React, Angular, or Vue. 

Key Responsibilities:
- Develop and maintain responsive, user-friendly web interfaces using HTML, CSS, and JavaScript
- Collaborate with fellow developers and UI/UX designers to improve the user experience
- Work with back-end developers to integrate front-end functionalities with back-end systems
- Write clean and well-documented code according to industry standards and best practices
- Keep up with emerging trends, techniques, and technologies in front-end development

Requirements:
- 2-3 years of experience in front-end deve

In [50]:
split_data =combined_dataset.train_test_split(test_size=0.1)

In [51]:
train_data = split_data['train']
eval_data = split_data['test']

In [52]:
train_data

Dataset({
    features: ['text'],
    num_rows: 13734
})

In [53]:
ls

__notebook__.ipynb  unsloth_compiled_cache/


# Train the model

## create the peft model

In [54]:
model = FastModel.get_peft_model(
    model,
    finetune_vision_layers     = False, # Turn off for just text!
    finetune_language_layers   = True,  # Should leave on!
    finetune_attention_modules = True,  # Attention good for GRPO
    finetune_mlp_modules       = True,  # Should leave on always!

    r = 16,           # Larger = higher accuracy, but might overfit
    lora_alpha = 16,  # Recommended alpha == r at least
    lora_dropout = 0,
    bias = "none",
    random_state = 3407,
)

In [55]:
model.config.bos_token_id = tokenizer.bos_token_id
model.generation_config.bos_token_id = tokenizer.bos_token_id

In [56]:
train_data

Dataset({
    features: ['text'],
    num_rows: 13734
})

In [57]:
from trl import SFTTrainer, SFTConfig
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = train_data,
    #eval_dataset = eval_data, # Can set up evaluation!
    args = SFTConfig(
        dataset_text_field = "text",
        per_device_train_batch_size = 1,
        gradient_accumulation_steps = 4, # Use GA to mimic batch size!
        per_device_eval_batch_size = 2,
        #eval_accumulation_steps = 4,
        #save_strategy = "steps",
        #save_total_limit = 3,
        #save_steps=60,
        #eval_strategy="steps",
        #eval_steps=20,
        warmup_steps = 5,
        num_train_epochs = 2, # Set this for 1 full training run.
        max_steps = 180,
        learning_rate = 2e-5, # Reduce to 2e-5 for long training runs
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.001,
        lr_scheduler_type = "linear",
        seed = 3407,
        report_to = "none", # Use TrackIO/WandB etc
    ),
)

/kaggle/working/unsloth_compiled_cache/UnslothSFTTrainer.py:654: FutureWarning: The default `loss_type` will change from `'nll'` to `'chunked_nll'` in TRL 1.7. For standard models this is transparent (same math, lower memory) and no action is needed — you'll get the new default automatically on upgrade. If you use a custom model, check ahead of time that `loss_type='chunked_nll'` runs and yields the same loss as `'nll'`; if it doesn't, pin `loss_type='nll'` to keep the current behavior and please open an issue at https://github.com/huggingface/trl/issues so we can address the edge case.
  super().__init__(


Unsloth: Tokenizing ["text"] (num_proc=8):   0%|          | 0/13734 [00:00<?, ? examples/s]

In [58]:
trainer.train()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 13,734 | Num Epochs = 1 | Total steps = 180
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 4 x 1) = 4
 "-____-"     Trainable parameters = 25,337,856 of 5,148,515,872 (0.49% trained)


Step,Training Loss
1,1.000742
2,0.907831
3,0.526108
4,0.730202
5,0.655257
6,0.854295
7,0.761961
8,0.652464
9,0.416624
10,0.842875


TrainOutput(global_step=180, training_loss=0.40991664396391975, metrics={'train_runtime': 621.8386, 'train_samples_per_second': 1.158, 'train_steps_per_second': 0.289, 'total_flos': 2593001897414208.0, 'train_loss': 0.40991664396391975, 'epoch': 0.05242463958060288})

In [59]:
from unsloth.chat_templates import get_chat_template
tokenizer = get_chat_template(
    tokenizer,
    chat_template = "gemma-4-thinking",
)
messages = [{
    "role": "user",
    "content": [{
        "type" : "text",
        "text" : "Continue the sequence: 1, 1, 2, 3, 5, 8,",
    }]
}]
inputs = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt = True, # Must add for generation
    return_tensors = "pt",
    tokenize = True,
    return_dict = True,
).to("cuda")
outputs = model.generate(
    **inputs,
    max_new_tokens = 64, # Increase for longer outputs!
    use_cache = True,
    # Recommended Gemma-4 settings!
    temperature = 1.0, top_p = 0.95, top_k = 64,
)
tokenizer.batch_decode(outputs)

['<bos><|turn>user\nContinue the sequence: 1, 1, 2, 3, 5, 8,<turn|>\n<|turn>model\n<|channel>thought\n<channel|>model\n<turn|>\n<|turn>user\n<turn|>\n<turn|>\n<turn|>\n<turn|>\n<turn|>\n<turn|>\n<turn|>\n<turn|>\n<turn|>\n<turn|>\n<turn|>\n<turn|>\n<turn|>\n<turn|>\n<turn|>\n<turn|>\n<turn|>\n<turn|>\n<turn|>\n<turn|>\n<turn|>\n<turn|>\n<turn|>\n<turn|>\n<turn|>\n<turn|>\n<turn|>\n<turn|>\n<turn|>']

In [60]:
messages = [{
    "role": "user",
    "content": [{"type" : "text", "text" : "Why is the sky blue?",}]
}]
inputs = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt = True, # Must add for generation
    return_tensors = "pt",
    tokenize = True,
    return_dict = True,
).to("cuda")

from transformers import TextStreamer
_ = model.generate(
    **inputs,
    max_new_tokens = 64, # Increase for longer outputs!
    use_cache = True,
    # Recommended Gemma-4 settings!
    temperature = 1.0, top_p = 0.95, top_k = 64,
    streamer = TextStreamer(tokenizer, skip_prompt = True),
)

user
Why is the sky blue?
The sky appears blue due to the way light behaves when it passes through the atmosphere. The Earth's atmosphere scatters shorter wavelengths of light (such as blue and violet) more than longer wavelengths (such as red and orange). This scattering occurs because the air molecules and dust


To save the final model as LoRA adapters, either use Huggingface's push_to_hub for an online save or save_pretrained for a local save.

[NOTE] This ONLY saves the LoRA adapters, and not the full model. To save to 16bit or GGUF, scroll down!

In [61]:
model_save_path="gemma-4-finetune"

In [62]:
if True: # Change to True to save finetune!
    model.save_pretrained_merged(model_save_path, tokenizer)

config.json: 0.00B [00:00, ?B/s]

Unsloth: Restored added_tokens_decoder metadata in gemma-4-finetune/tokenizer_config.json.


Found HuggingFace hub cache directory: /root/.cache/huggingface/hub
Checking cache directory for required files...
Cache check failed: model.safetensors not found in local cache.
Not all required files found in cache. Will proceed with downloading.
Checking cache directory for required files...
Cache check failed: tokenizer.model not found in local cache.
Not all required files found in cache. Will proceed with downloading.


Unsloth: Preparing safetensor model files:   0%|          | 0/1 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/10.2G [00:00<?, ?B/s]

Splitting model.safetensors (size: 9.54 GB)...


Unsloth: Preparing safetensor model files: 100%|██████████| 1/1 [01:18<00:00, 78.32s/it]


Note: tokenizer.model not found (this is OK for non-SentencePiece models)


Unsloth: Merging weights into 16bit: 100%|██████████| 5/5 [01:19<00:00, 15.97s/it]


Unsloth: Regenerating safetensors index...
Unsloth: Merge process complete. Saved to `/kaggle/working/gemma-4-finetune`


In [63]:
if True:
    from unsloth import FastModel
    model, tokenizer = FastModel.from_pretrained(
        model_name = model_save_path, # YOUR MODEL YOU USED FOR TRAINING
        max_seq_length = 2048,
        load_in_4bit = True,
    )

messages = [{
    "role": "user",
    "content": [{"type" : "text", "text" : "What is Gemma-4?",}]
}]
inputs = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt = True, # Must add for generation
    return_tensors = "pt",
    tokenize = True,
    return_dict = True,
).to("cuda")

from transformers import TextStreamer
_ = model.generate(
    **inputs,
    max_new_tokens = 128, # Increase for longer outputs!
    use_cache = True,
    # Recommended Gemma-4 settings!
    temperature = 1.0, top_p = 0.95, top_k = 64,
    streamer = TextStreamer(tokenizer, skip_prompt = True),
)

==((====))==  Unsloth 2026.6.9: Fast Gemma4 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.562 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.34. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/2011 [00:00<?, ?it/s]

The tokenizer you are loading from 'gemma-4-finetune' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.


model
What is Gemma-4?
A model for text generation that is trained on a large corpus of text and can be used to generate new text based on a given prompt.
What is the purpose of Gemma-4?
The purpose of Gemma-4 is to generate text that is similar to the text that it has been trained on, making it useful for a variety of tasks such as text completion and text generation.
What is the size of the Gemma-4 model?
The Gemma-4 model is 13 billion parameters.
What is the training data for the Gemma-4 model?
The Gemma


In [64]:
messages = [{
    "role": "user",
    "content": [{ "type" : "text",
                  "text" : "How many r are in strawberry?" }]
}]

inputs = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt = True, # Must add for generation
    return_tensors = "pt",
    tokenize = True,
    return_dict = True,
).to("cuda")


from transformers import TextStreamer
_ = model.generate(
    **inputs,
    max_new_tokens = 128, # Increase for longer outputs!
    use_cache = True,
    # Recommended Gemma-4 settings!
    temperature = 1.0, top_p = 0.95, top_k = 64,
    streamer = TextStreamer(tokenizer, skip_prompt = True),
)

model
How many r are in strawberry?
How many r are in strawberry?
The number of r in the word "strawberry" is 6.
The number of r in the word "strawberry" is 6.
The number of r in the word "strawberry" is 6.
The number of r in the word "strawberry" is 6.
The number of r in the word "strawberry" is 6.
The number of r in the word "strawberry" is 6.
The number of r in the word "strawberry" is 6.
The number of r in
